# Chatbot de Reservas Basado en Reglas — Nivel 0 de la Progresión de Agentes

---

**Autor:** Borja Mora Méndez
**Contacto:** [borja.mora.mendez@gmail.com](mailto:borja.mora.mendez@gmail.com) · [LinkedIn](https://www.linkedin.com/in/borja-mora-mendez/)
**Repositorio:** [Data Analytics Portfolio](https://github.com/BORJAMOME/Data-Analytics-Portfolio)
**Categoría:** IA & Big Data · Agentes IA · Nivel 0 (reglas, sin IA)

---

### Objetivo

Construir un sistema de reservas para un restaurante controlado por un menú de opciones y
reglas de negocio explícitas (aforo máximo, validación de fecha/hora), como **línea base
sin IA** de la categoría Agentes IA.

### Contexto de negocio

**El cliente:** un restaurante que necesita gestionar reservas sin sobrepasar su aforo.

**El problema:** antes de plantearse IA, similitud semántica o LLMs, hay que preguntarse si
el problema realmente los necesita. Aquí el "chatbot" no interpreta lenguaje natural: el
usuario elige una opción numérica y el programa ejecuta la función de reglas correspondiente.

**La pregunta:** ¿cuánta inteligencia hace falta realmente para resolver esto? Este notebook
es intencionadamente el **Nivel 0** de la categoría: la línea base de "cero inteligencia"
frente a la que se comparan los siguientes casos, donde el mismo tipo de problema —
interpretar qué quiere el usuario— se resuelve primero con similitud semántica (sin LLM) y
después con un LLM local, para medir qué aporta realmente cada capa de sofisticación.

In [ ]:
import pandas as pd

## 1. Configuración inicial — aforo y registro de reservas

In [ ]:
# Aforo máximo del restaurante
AFORO_MAXIMO = 50

# DataFrame en memoria con las reservas activas
reservas = pd.DataFrame(
    columns=[
        "Nombre",
        "Fecha",
        "Hora",
        "Personas"
    ]
)

## 2. Motor de reglas — funciones de negocio

In [ ]:
def personas_reservadas(fecha, hora):
    if reservas.empty:
        return 0

    filtro = (
        (reservas["Fecha"] == fecha)
        & (reservas["Hora"] == hora)
    )

    return reservas.loc[filtro, "Personas"].sum()


def hacer_reserva():
    global reservas

    nombre = input("Ingrese su nombre: ")
    fecha = input("Fecha (AAAA-MM-DD): ")
    hora = input("Hora (HH:MM): ")
    personas = int(input("Número de personas: "))

    ocupadas = personas_reservadas(fecha, hora)

    if ocupadas + personas <= AFORO_MAXIMO:

        nueva = pd.DataFrame({
            "Nombre": [nombre],
            "Fecha": [fecha],
            "Hora": [hora],
            "Personas": [personas]
        })

        reservas = pd.concat(
            [reservas, nueva],
            ignore_index=True
        )

        print("Reserva realizada con éxito.")

    else:
        print("No hay capacidad disponible.")


def consultar_disponibilidad():
    fecha = input("Fecha (AAAA-MM-DD): ")
    hora = input("Hora (HH:MM): ")

    ocupadas = personas_reservadas(fecha, hora)
    libres = AFORO_MAXIMO - ocupadas

    print(
        f"Disponibilidad para {fecha} a las {hora}: "
        f"{libres} personas."
    )


def cancelar_reserva():
    global reservas

    nombre = input("Ingrese su nombre: ")

    if nombre in reservas["Nombre"].values:

        reservas = reservas[
            reservas["Nombre"] != nombre
        ]

        print("Reserva cancelada.")

    else:
        print("No se encontró ninguna reserva con ese nombre.")


def mostrar_reservas():
    if reservas.empty:
        print("No hay reservas.")

    else:
        print("\nReservas actuales:")
        print(reservas)

## 3. Bucle conversacional basado en menú de opciones

El "chatbot" no interpreta lenguaje natural: el usuario elige una opción numérica y el
programa ejecuta la función de reglas correspondiente.

In [ ]:
def chatbot():

    print("=" * 50)
    print("Bienvenido al sistema de reservas del restaurante.")
    print("=" * 50)

    while True:

        print("\nOpciones:")
        print("1. Hacer una reserva")
        print("2. Consultar disponibilidad")
        print("3. Cancelar una reserva")
        print("4. Mostrar todas las reservas")
        print("5. Salir")

        opcion = input("Seleccione una opción (1-5): ")

        if opcion == "1":
            hacer_reserva()

        elif opcion == "2":
            consultar_disponibilidad()

        elif opcion == "3":
            cancelar_reserva()

        elif opcion == "4":
            mostrar_reservas()

        elif opcion == "5":
            print(
                "Gracias por usar el sistema de reservas. "
                "¡Hasta luego!"
            )
            break

        else:
            print(
                "Opción no válida. "
                "Por favor, intente de nuevo."
            )


# Descomentar para ejecutar en modo interactivo (requiere terminal con input()):
# chatbot()

## 4. Demostración — el sistema en funcionamiento

`chatbot()` depende de `input()`, así que para demostrar el comportamiento de forma
reproducible (sin depender de una consola interactiva) simulamos aquí la misma lógica de
negocio con reservas de ejemplo.

In [ ]:
# Simulamos 3 reservas para el mismo turno, probando el control de aforo
reservas_ejemplo = [
    ("Ana García", "2026-09-12", "21:00", 4),
    ("Luis Pérez", "2026-09-12", "21:00", 6),
    ("Marta Ruiz", "2026-09-12", "21:30", 2),
]

for nombre, fecha, hora, personas in reservas_ejemplo:
    ocupadas = personas_reservadas(fecha, hora)

    if ocupadas + personas <= AFORO_MAXIMO:
        nueva = pd.DataFrame({
            "Nombre": [nombre],
            "Fecha": [fecha],
            "Hora": [hora],
            "Personas": [personas]
        })
        reservas = pd.concat([reservas, nueva], ignore_index=True)
        print(f"Reserva realizada con éxito para {nombre}.")
    else:
        print(f"No hay capacidad disponible para {nombre}.")

mostrar_reservas()

Reserva realizada con éxito para Ana García.
Reserva realizada con éxito para Luis Pérez.
Reserva realizada con éxito para Marta Ruiz.

Reservas actuales:
       Nombre       Fecha   Hora Personas
0  Ana García  2026-09-12  21:00        4
1  Luis Pérez  2026-09-12  21:00        6
2  Marta Ruiz  2026-09-12  21:30        2


In [ ]:
# Consultamos disponibilidad para el turno de las 21:00, ya con 10 personas reservadas
ocupadas = personas_reservadas("2026-09-12", "21:00")
libres = AFORO_MAXIMO - ocupadas

print(f"Disponibilidad para 2026-09-12 a las 21:00: {libres} personas.")

Disponibilidad para 2026-09-12 a las 21:00: 40 personas.


## 5. Conclusión

**El hallazgo:** un menú de opciones con reglas explícitas resuelve perfectamente el
problema de gestionar reservas con control de aforo — no todo lo que se llama "chatbot"
necesita IA. La complejidad debe justificarse por el problema, no añadirse por defecto.

**Por qué importa como línea base:** los siguientes casos de esta categoría resuelven un
problema distinto y más difícil — interpretar una pregunta en **lenguaje libre**, no una
opción de menú — primero con similitud semántica (sin LLM) y después con un LLM local. Este
Nivel 0 es el punto de comparación: si esas capas de sofisticación no mejoran la experiencia
frente a un menú simple, no están justificadas.